In [16]:
#Base Model Architecture & Pre-Training (Region A)

import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load your pre-trained model trained on Region A
base_model = tf.keras.models.load_model('model_region_A.keras')

# Display architecture summary
base_model.summary()

Model: "Base_Model_Region_A"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_layer_1 (LSTM)             │ (None, 4, 64)          │        19,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer_2 (LSTM)             │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_feature_head (Dense)      │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,789 (385.90 KB)

 Trainable params: 32,929 (128.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 65,860 (257.27 KB)

In [21]:
#Data Preprocessing & Sequence Preparation

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Define features and target variable
FEATURES = [
    'DC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION',
    'hour', 'minute', 'day_of_week', 'day_of_year', 'month',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
TARGET = 'AC_POWER'
SEQUENCE_LENGTH = 4  # 4 time-steps (e.g., 1 hour of 15-min intervals)

def load_and_create_sequences(train_path, val_path, test_path, seq_len=SEQUENCE_LENGTH):
    """Loads CSV sets and constructs 3D sliding window tensors (samples, timesteps, features)."""
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)

    def create_sequences(df, seq_length):
        X, y = [], []
        data_x = df[FEATURES].values
        data_y = df[TARGET].values
        for i in range(len(df) - seq_length):
            X.append(data_x[i : i + seq_length])
            y.append(data_y[i + seq_length])
        return np.array(X), np.array(y)

    X_train, y_train = create_sequences(df_train, seq_len)
    X_val, y_val = create_sequences(df_val, seq_len)
    X_test, y_test = create_sequences(df_test, seq_len)

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

# Load Region A Datasets
(X_train_A, y_train_A), (X_val_A, y_val_A), (X_test_A, y_test_A) = load_and_create_sequences(
    'train.csv', 'validation.csv', 'test.csv'
)

# Load Region B Datasets
(X_train_B, y_train_B), (X_val_B, y_val_B), (X_test_B, y_test_B) = load_and_create_sequences(
    'train (1).csv', 'validation (1).csv', 'test (1).csv'
)

print(f"Region A Train shape: {X_train_A.shape}, Test shape: {X_test_A.shape}")
print(f"Region B Train shape: {X_train_B.shape}, Test shape: {X_test_B.shape}")

Region A Train shape: (2206, 4, 13), Test shape: (470, 4, 13)
Region B Train shape: (2277, 4, 13), Test shape: (485, 4, 13)


In [20]:
#Build & Pre-train Base Model on Region A

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

# 1. Build Base Architecture
base_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    LSTM(64, return_sequences=True, name="lstm_layer_1"),
    Dropout(0.2),
    LSTM(32, return_sequences=False, name="lstm_layer_2"),
    Dropout(0.2),
    Dense(16, activation='relu', name="dense_feature_head"),
    Dense(1, activation='linear', name="output_layer")
], name="Base_Model_Region_A")

base_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

# 2. Train on Region A
print("\n--- Pre-training Base Model on Region A ---")
base_model.fit(
    X_train_A, y_train_A,
    validation_data=(X_val_A, y_val_A),
    epochs=25,
    batch_size=32,
    verbose=1
)

# Save Region A Base Weights
base_model.save('model_region_A.keras')


--- Pre-training Base Model on Region A ---
Epoch 1/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 124092816.0000 - mae: 6862.5723 - val_loss: 107014424.0000 - val_mae: 6400.6729
Epoch 2/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 123880392.0000 - mae: 6861.4644 - val_loss: 106793736.0000 - val_mae: 6399.4297
Epoch 3/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 123613912.0000 - mae: 6859.9863 - val_loss: 106513960.0000 - val_mae: 6397.8398
Epoch 4/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 123279336.0000 - mae: 6858.9048 - val_loss: 106154960.0000 - val_mae: 6395.7900
Epoch 5/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 122853344.0000 - mae: 6856.2354 - val_loss: 105723128.0000 - val_mae: 6393.3140
Epoch 6/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 122346864.0000 - mae: 6853.9341 - val_loss: 105219976.0000 - val_mae: 6390.5352
Epoch 7/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 121760792.0000 - mae: 6852.0640 - val_loss: 104636752.0000 - val_mae:

In [19]:
#Layer Freezing & Transfer Learning (Region B)

# Load pre-trained Region A model
transfer_model_frozen = tf.keras.models.load_model('model_region_A.keras')

# Freeze feature extraction layers (keep only top classification/regression heads trainable)
for layer in transfer_model_frozen.layers[:-2]:
    layer.trainable = False

transfer_model_frozen.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

print("\n--- Training Frozen Feature Extractor on Region B ---")
transfer_model_frozen.fit(
    X_train_B, y_train_B,
    validation_data=(X_val_B, y_val_B),
    epochs=15,
    batch_size=32,
    verbose=1
)


--- Training Frozen Feature Extractor on Region B ---
Epoch 1/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 55213048.0000 - mae: 5249.6934 - val_loss: 44072636.0000 - val_mae: 4718.0806
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 55139944.0000 - mae: 5253.9541 - val_loss: 44019456.0000 - val_mae: 4718.8735
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 55024928.0000 - mae: 5255.6211 - val_loss: 43964176.0000 - val_mae: 4719.7012
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 55040476.0000 - mae: 5257.5088 - val_loss: 43911644.0000 - val_mae: 4720.5000
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55014136.0000 - mae: 5259.5776 - val_loss: 43857016.0000 - val_mae: 4721.3608
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 54819176.0000 - mae: 5252.4097 - val_loss: 43803836.0000 - val_mae: 4722.2134
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 54786612.0000 - mae: 5257.0371 - val_loss: 43749544.0000 - val_mae: 472

In [18]:
#Fine-Tuning Stage (Unfreezing Upper Layers)

# Clone model for fine-tuning
fine_tune_model = tf.keras.models.load_model('model_region_A.keras')

# Unfreeze upper LSTM layer and Dense layers for subtle weight updates
for layer in fine_tune_model.layers:
    if layer.name in ['lstm_layer_2', 'dense_feature_head', 'output_layer']:
        layer.trainable = True
    else:
        layer.trainable = False

# Recompile with reduced learning rate to prevent exploding/distorting pre-trained features
fine_tune_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='mse',
    metrics=['mae']
)

print("\n--- Fine-tuning Unfrozen Model on Region B ---")
fine_tune_model.fit(
    X_train_B, y_train_B,
    validation_data=(X_val_B, y_val_B),
    epochs=20,
    batch_size=32,
    verbose=1
)


--- Fine-tuning Unfrozen Model on Region B ---
Epoch 1/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 55348480.0000 - mae: 5261.3296 - val_loss: 44121896.0000 - val_mae: 4717.3716
Epoch 2/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55253628.0000 - mae: 5253.5493 - val_loss: 44116448.0000 - val_mae: 4717.4478
Epoch 3/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55252096.0000 - mae: 5253.9775 - val_loss: 44110960.0000 - val_mae: 4717.5264
Epoch 4/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55271940.0000 - mae: 5252.9521 - val_loss: 44105696.0000 - val_mae: 4717.6006
Epoch 5/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55180224.0000 - mae: 5250.7993 - val_loss: 44100288.0000 - val_mae: 4717.6768
Epoch 6/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55247560.0000 - mae: 5252.4336 - val_loss: 44094824.0000 - val_mae: 4717.7549
Epoch 7/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 55158368.0000 - mae: 5250.8198 - val_loss: 44089496.0000 - val_mae: 4717.8311


In [17]:
#Comparative Benchmarking

# 1. Baseline Model (Trained strictly on Region B from Scratch)
scratch_model_B = tf.keras.models.clone_model(base_model)
scratch_model_B.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)
scratch_model_B.fit(
    X_train_B, y_train_B,
    validation_data=(X_val_B, y_val_B),
    epochs=25,
    batch_size=32,
    verbose=0
)

# 2. Benchmark Evaluation Function
def evaluate_strategy(model, X_test, y_test, strategy_name):
    preds = model.predict(X_test, verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    return {
        "Strategy": strategy_name,
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "R² Score": round(r2, 4)
    }

# 3. Aggregate Performance Metrics on Region B Test Data (test (1).csv)
results = [
    evaluate_strategy(scratch_model_B, X_test_B, y_test_B, "Scratch Model (Region B Only)"),
    evaluate_strategy(transfer_model_frozen, X_test_B, y_test_B, "Layer Freezing (Feature Extractor)"),
    evaluate_strategy(fine_tune_model, X_test_B, y_test_B, "Fine-Tuned Transfer Model")
]

benchmark_df = pd.DataFrame(results)
print("\n--- Comparative Benchmarking Summary (Region B Test Set) ---")
print(benchmark_df.to_string(index=False))


--- Comparative Benchmarking Summary (Region B Test Set) ---
                          Strategy      RMSE       MAE  R² Score
     Scratch Model (Region B Only) 5782.4115 3412.2034   -0.0501
Layer Freezing (Feature Extractor) 6256.9229 4548.4227   -0.2296
         Fine-Tuned Transfer Model 6309.7195 4538.3033   -0.2504
